In [0]:
# CRUD Operations Demo untuk Databricks
# Proof of Concept: REST API style functions

from pyspark.sql import Row
from datetime import datetime
import pandas as pd

print("✅ Setup complete!")
print("📊 Table: workspace.default.bronze_closing_transaction")
print("🔧 CRUD functions ready to use")

In [0]:
# ==================== READ OPERATIONS ====================

def get_transactions(limit=10, offset=0, filters=None):
    """
    GET /transactions - Fetch paginated transactions
    
    Parameters:
    - limit: jumlah data per page (default: 10)
    - offset: skip data (untuk pagination)
    - filters: dict untuk WHERE clause, contoh: {'truck': 'T001', 'driver': 'John'}
    """
    where_clause = ""
    if filters:
        conditions = [f"{k} = '{v}'" if isinstance(v, str) else f"{k} = {v}" 
                     for k, v in filters.items()]
        where_clause = "WHERE " + " AND ".join(conditions)
    
    query = f"""
        SELECT * FROM workspace.default.bronze_closing_transaction
        {where_clause}
        ORDER BY id
        LIMIT {limit} OFFSET {offset}
    """
    
    df = spark.sql(query)
    result = df.toPandas()
    
    # Get total count
    count_query = f"SELECT COUNT(*) as total FROM workspace.default.bronze_closing_transaction {where_clause}"
    total = spark.sql(count_query).collect()[0]['total']
    
    print(f"📄 Showing {len(result)} of {total} total records")
    return result


def get_transaction_by_id(transaction_id):
    """
    GET /transactions/{id} - Fetch single transaction
    """
    query = f"""
        SELECT * FROM workspace.default.bronze_closing_transaction
        WHERE id = {transaction_id}
    """
    
    df = spark.sql(query)
    result = df.toPandas()
    
    if len(result) == 0:
        print(f"❌ Transaction ID {transaction_id} not found")
        return None
    
    print(f"✅ Found transaction ID {transaction_id}")
    return result


# Test READ operations
print("\n🔍 Testing READ operations...\n")
print("1️⃣ Get first 5 transactions:")
display(get_transactions(limit=5))

print("\n2️⃣ Get transaction by ID (ID=1):")
display(get_transaction_by_id(1))

In [0]:
# ==================== TEST DENGAN DATA ASLI ====================

print("🔍 Testing READ operations with real data...\n")
print("="*60)

# 1. Test get_transaction_by_id dengan ID yang benar
print("\n1️⃣ GET single transaction by actual ID:")
print("-" * 60)
actual_id = 101397830170138498  # ID dari data pertama
result = get_transaction_by_id(actual_id)

if result is not None:
    display(result)
    print(f"\n✅ Successfully fetched transaction ID: {actual_id}")
    print(f"   Truck: {result['truck'].iloc[0]}")
    print(f"   Driver: {result['driver'].iloc[0]}")
    print(f"   Client: {result['client'].iloc[0]}")
    print(f"   Gross: {result['gross'].iloc[0]} kg")
    print(f"   Netto: {result['netto'].iloc[0]} kg")

# 2. Test pagination
print("\n\n2️⃣ GET with pagination (Page 2 - skip first 5, get next 3):")
print("-" * 60)
page_2 = get_transactions(limit=3, offset=5)
display(page_2)

# 3. Test filtering by client
print("\n\n3️⃣ GET with filter (Client = 'KUSAN'):")
print("-" * 60)
filtered_kusan = get_transactions(limit=5, filters={'client': 'KUSAN'})
display(filtered_kusan)

# 4. Test filtering by truck
print("\n\n4️⃣ GET with filter (Truck = 'RAM 8070'):")
print("-" * 60)
filtered_truck = get_transactions(limit=5, filters={'truck': 'RAM 8070'})
display(filtered_truck)

# 5. Summary statistics
print("\n\n📊 Data Summary:")
print("="*60)
all_data = get_transactions(limit=100)
print(f"  • Total records in table: 29,787")
print(f"  • Sample size fetched: {len(all_data)}")
print(f"  • Unique clients: {all_data['client'].nunique()}")
print(f"  • Unique trucks: {all_data['truck'].nunique()}")
print(f"  • Unique drivers: {all_data['driver'].nunique()}")
print(f"  • Average Gross weight: {all_data['gross'].mean():.2f} kg")
print(f"  • Average Netto weight: {all_data['netto'].mean():.2f} kg")

print("\n✅ All READ operations tested successfully!")
print("\n💡 Your data is safe - no modifications made!")

In [0]:
# ==================== CREATE OPERATION ====================

def create_transaction(truck, driver, client, material=None, hauling=None, coal=None, 
                       code=None, gross=None, netto=None, tare=None, mode=None):
    """
    POST /transactions - Insert new transaction
    
    Required: truck, driver, client
    Optional: material, hauling, coal, code, gross, netto, tare, mode
    """
    # Build INSERT query
    columns = ['truck', 'driver', 'client']
    values = [f"'{truck}'", f"'{driver}'", f"'{client}'"]
    
    optional_fields = {
        'material': material,
        'hauling': hauling,
        'coal': coal,
        'code': code,
        'gross': gross,
        'netto': netto,
        'tare': tare,
        'mode': mode
    }
    
    for field, value in optional_fields.items():
        if value is not None:
            columns.append(field)
            if isinstance(value, str):
                values.append(f"'{value}'")
            else:
                values.append(str(value))
    
    query = f"""
        INSERT INTO workspace.default.bronze_closing_transaction 
        ({', '.join(columns)})
        VALUES ({', '.join(values)})
    """
    
    spark.sql(query)
    print(f"✅ Transaction created: {truck} - {driver} - {client}")
    
    # Get the newly created record
    last_id = spark.sql("SELECT MAX(id) as max_id FROM workspace.default.bronze_closing_transaction").collect()[0]['max_id']
    return get_transaction_by_id(last_id)


# ==================== UPDATE OPERATION ====================

def update_transaction(transaction_id, **kwargs):
    """
    PUT /transactions/{id} - Update existing transaction
    
    Example: update_transaction(1, driver='New Driver', gross=2500.0)
    """
    if not kwargs:
        print("❌ No fields to update")
        return None
    
    # Check if transaction exists
    existing = get_transaction_by_id(transaction_id)
    if existing is None:
        return None
    
    # Build SET clause
    set_clauses = []
    for key, value in kwargs.items():
        if isinstance(value, str):
            set_clauses.append(f"{key} = '{value}'")
        else:
            set_clauses.append(f"{key} = {value}")
    
    query = f"""
        UPDATE workspace.default.bronze_closing_transaction
        SET {', '.join(set_clauses)}
        WHERE id = {transaction_id}
    """
    
    spark.sql(query)
    print(f"✅ Transaction ID {transaction_id} updated")
    
    # Show updated record
    return get_transaction_by_id(transaction_id)


# ==================== DELETE OPERATION ====================

def delete_transaction(transaction_id):
    """
    DELETE /transactions/{id} - Delete transaction
    """
    # Check if transaction exists
    existing = get_transaction_by_id(transaction_id)
    if existing is None:
        return False
    
    query = f"""
        DELETE FROM workspace.default.bronze_closing_transaction
        WHERE id = {transaction_id}
    """
    
    spark.sql(query)
    print(f"✅ Transaction ID {transaction_id} deleted")
    return True


print("✅ All CRUD functions loaded!")
print("\n📝 Available functions:")
print("  • get_transactions(limit, offset, filters)")
print("  • get_transaction_by_id(id)")
print("  • create_transaction(truck, driver, client, ...)")
print("  • update_transaction(id, **kwargs)")
print("  • delete_transaction(id)")

In [0]:
# ==================== COMPLETE CRUD DEMO ====================

print("🚀 Starting CRUD Operations Demo...\n")
print("="*60)

# 1. CREATE - Insert new transaction
print("\n1️⃣ CREATE: Inserting new transaction...")
print("-" * 60)
new_record = create_transaction(
    truck="TRUCK-API-TEST",
    driver="API Test Driver",
    client="Test Client Corp",
    material="Coal",
    gross=2500.5,
    netto=2200.0,
    tare=300.5,
    mode=1
)
display(new_record)

# Get the new ID
new_id = new_record['id'].iloc[0]
print(f"\n✅ New transaction created with ID: {new_id}")

# 2. READ - Get the created transaction
print("\n2️⃣ READ: Fetching transaction by ID...")
print("-" * 60)
fetched = get_transaction_by_id(new_id)
display(fetched)

# 3. UPDATE - Modify the transaction
print("\n3️⃣ UPDATE: Modifying transaction...")
print("-" * 60)
updated = update_transaction(
    new_id,
    driver="Updated Driver Name",
    gross=2800.0,
    netto=2500.0
)
display(updated)

# 4. READ with pagination
print("\n4️⃣ READ: Paginated list (last 10 transactions)...")
print("-" * 60)
recent_transactions = get_transactions(limit=10, offset=0)
display(recent_transactions)

# 5. READ with filters
print("\n5️⃣ READ: Filtered search (truck = 'TRUCK-API-TEST')...")
print("-" * 60)
filtered = get_transactions(limit=5, filters={'truck': 'TRUCK-API-TEST'})
display(filtered)

# 6. DELETE - Remove the test transaction
print("\n6️⃣ DELETE: Removing test transaction...")
print("-" * 60)
delete_result = delete_transaction(new_id)

if delete_result:
    print(f"\n✅ Transaction ID {new_id} has been deleted")
    
    # Verify deletion
    print("\n🔍 Verifying deletion...")
    verify = get_transaction_by_id(new_id)
    if verify is None or len(verify) == 0:
        print("✅ Deletion confirmed - record no longer exists")

print("\n" + "="*60)
print("✅ CRUD Demo Complete!")
print("\n📊 Summary:")
print("  ✅ CREATE - Insert new record")
print("  ✅ READ - Fetch by ID and pagination")
print("  ✅ UPDATE - Modify existing record")
print("  ✅ DELETE - Remove record")
print("\n🎉 All operations working successfully!")
print("\n💡 Tip: You can now test these functions with your own data!")
print("   Example: create_transaction('T001', 'John', 'ABC Corp', material='Steel')")